In [2]:
!pip install -U docling langchain langchain-postgres psycopg[binary] langchain-huggingface sentence-transformers langgraph langchain-text-splitters

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 263.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.2/625.2 kB 661.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 231.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 228.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 443.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 226.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 303.9 MB/s  0:00:01eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 306.6 MB/s  0:00:01eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 317.1 MB/s  0:00:00eta 0:00:01
   ━

In [3]:
import os
from langchain_postgres.vectorstores import PGVector
from langchain_huggingface import HuggingFaceEmbeddings

print("⏳ Initializing models and connecting to database...")

# 1. Load the URL
db_url = os.environ.get("DATABASE_URL")

# 2. Load the tiny embedding model
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 3. Attempt the connection
try:
    vector_store = PGVector(
        embeddings=embeddings,
        collection_name="connection_test",
        connection=db_url,
    )
    print("✅ SUCCESS: Connected to CloudNativePG and initialized pgvector!")
except Exception as e:
    print(f"❌ FAILED: Could not connect to the database. Error: {e}")

⏳ Initializing models and connecting to database...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ SUCCESS: Connected to CloudNativePG and initialized pgvector!


In [4]:
import os
from langchain_postgres.vectorstores import PGVector
from langchain_huggingface import HuggingFaceEmbeddings

print("⏳ Initializing models and connecting to database...")

# 1. Load the URL
db_url = os.environ.get("DATABASE_URL")

# 2. Load the tiny embedding model
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 3. Attempt the connection
try:
    vector_store = PGVector(
        embeddings=embeddings,
        collection_name="connection_test",
        connection=db_url,
    )
    print("✅ SUCCESS: Connected to CloudNativePG and initialized pgvector!")
except Exception as e:
    print(f"❌ FAILED: Could not connect to the database. Error: {e}")

⏳ Initializing models and connecting to database...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ SUCCESS: Connected to CloudNativePG and initialized pgvector!


In [5]:
import logging
logging.basicConfig(level=logging.INFO)

In [6]:
import pandas as pd
from docling.document_converter import DocumentConverter
from langchain_core.documents import Document
from langchain_postgres.vectorstores import PGVector
from langchain_huggingface import HuggingFaceEmbeddings

print("📄 Parsing CDC Guidelines with Docling...")
converter = DocumentConverter()

# 1. Parse the PDF
# Docling automatically detects headers, paragraphs, and tables.
result = converter.convert("cdc_mec_tables_only.pdf")
doc = result.document

unrolled_documents = []

print(f"🔍 Found {len(doc.tables)} tables. Unrolling into semantic statements...")

# 2. Iterate ONLY through the detected tables
for table_ix, table in enumerate(doc.tables):
    
    # Docling cleanly exports the visual grid into a Pandas DataFrame
    df = table.export_to_dataframe()
    
    # Ensure the first column is our "Condition" column 
    # (You may need to tweak this depending on exactly how Docling reads the header row)
    if df.empty or len(df.columns) < 2:
        continue
        
    condition_col = df.columns[0]
    
    # Identify the contraceptive methods (the columns)
    # Exclude non-method columns like "Clarification" or "Condition"
    methods = [col for col in df.columns if col.strip() not in [condition_col, "Clarification", "None", ""]]
    
    # 3. Unroll the 2D matrix into 1D explicit statements
    for index, row in df.iterrows():
        condition = str(row[condition_col]).strip()
        clarification = str(row.get("Clarification", "")).strip()
        
        # Skip empty rows or artifact rows
        if not condition or condition.lower() in ["nan", "none", "condition"]:
            continue
            
        for method in methods:
            category_raw = str(row[method]).strip()
            
            # Extract just the first number (e.g., if it says "2/3" or "4*", just grab the integer)
            category = "".join([c for c in category_raw if c.isdigit()])
            
            if category:
                # 💥 THIS is the magic. We create a dense, context-rich string for every single cell.
                statement = (
                    f"According to the 2024 CDC Medical Eligibility Criteria (MEC), "
                    f"for a patient with the condition '{condition}', "
                    f"the safety category for using '{method}' is Category {category[0]}."
                )
                
                if clarification and clarification.lower() not in ["nan", "none"]:
                    statement += f" Clinical clarification: {clarification}"
                
                # 4. Create the LangChain Document with Metadata for Stage 4 filtering
                unrolled_documents.append(Document(
                    page_content=statement,
                    metadata={
                        "condition": condition.lower(),
                        "method": method.lower(),
                        "category_score": int(category[0]),
                        "source": "cdc_mec_docling"
                    }
                ))

print(f"✅ Generated {len(unrolled_documents)} highly structured chunks!")

# 5. Push to PGVector (Your existing logic)
# embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
# vector_store = PGVector( ... )
# vector_store.add_documents(unrolled_documents)

INFO:docling.datamodel.document:detected formats: [<InputFormat.PDF: 'pdf'>]
INFO:docling.document_converter:Going to convert document batch...
INFO:docling.document_converter:Initializing pipeline for StandardPdfPipeline with options hash a6a2eef09bfdbd6bdaeaf15fcaebde59
INFO:docling.models.factories.base_factory:Loading plugin 'docling_defaults'
INFO:docling.models.factories:Registered picture descriptions: ['picture_description_vlm_engine', 'vlm', 'api']
INFO:docling.models.factories.base_factory:Loading plugin 'docling_defaults'
INFO:docling.models.factories:Registered ocr engines: ['auto', 'easyocr', 'kserve_v2_ocr', 'ocrmac', 'rapidocr', 'tesserocr', 'tesseract']
INFO:docling.models.stages.ocr.auto_ocr_model:rapidocr cannot be used because onnxruntime is not installed.
INFO:docling.models.stages.ocr.auto_ocr_model:easyocr cannot be used because it is not installed.


📄 Parsing CDC Guidelines with Docling...


INFO:docling.utils.accelerator_utils:Accelerator device: 'cpu'
[INFO] 2026-03-31 18:47:49,996 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-03-31 18:47:49,999 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-03-31 18:47:50,000 [RapidOCR] download_file.py:68: Initiating download: https://www.modelscope.cn/models/RapidAI/RapidOCR/resolve/v3.7.0/torch/PP-OCRv4/det/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-03-31 18:47:52,033 [RapidOCR] download_file.py:82: Download size: 13.83MB
[INFO] 2026-03-31 18:47:52,353 [RapidOCR] download_file.py:95: Successfully saved to: /opt/app-root/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-03-31 18:47:52,355 [RapidOCR] main.py:50: Using /opt/app-root/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-03-31 18:47:52,518 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-03-31 18:47:52,519 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-03-31 18:47

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

INFO:docling.models.factories.base_factory:Loading plugin 'docling_defaults'
INFO:docling.models.factories:Registered table structure engines: ['docling_tableformer', 'docling_tableformer_v2']
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/docling-project/docling-models/revision/v2.3.0 "HTTP/1.1 200 OK"
INFO:docling.utils.accelerator_utils:Accelerator device: 'cpu'
INFO:docling.pipeline.base_pipeline:Processing document cdc_mec_tables_only.pdf
INFO:docling.document_converter:Finished converting document cdc_mec_tables_only.pdf in 642.91 sec.


🔍 Found 79 tables. Unrolling into semantic statements...


✅ Generated 3239 highly structured chunks!


In [7]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_postgres.vectorstores import PGVector

# 1. Initialize your embeddings model
print("🧠 Loading embedding model...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 2. Your database connection string (Make sure to use your actual OpenShift DB URL)
db_url = os.environ.get("DATABASE_URL")

# 3. Connect and WIPE the old data
print("🗑️ Wiping old fragmented RAG tables...")
vector_store = PGVector(
    embeddings=embeddings,
    collection_name="cdc_mec_rules",
    connection=db_url,
    use_jsonb=True, 
)
vector_store.drop_tables() 

# 4. Re-initialize and upload the NEW unrolled chunks
print("🏗️ Rebuilding fresh tables and collection...")
vector_store = PGVector(
    embeddings=embeddings,
    collection_name="cdc_mec_rules",
    connection=db_url,
    use_jsonb=True, 
)

print(f"💾 Uploading {len(unrolled_documents)} structured chunks to the database...")
vector_store.add_documents(unrolled_documents)

print("✅ SUCCESS: The fully optimized RAG data is loaded into CloudNativePG!")

INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: all-MiniLM-L6-v2
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config_sentence_transformers.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve

🧠 Loading embedding model...


INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/sentence_bert_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/sentence_bert_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config.json "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/tokenizer_config.json "HT

🗑️ Wiping old fragmented RAG tables...
🏗️ Rebuilding fresh tables and collection...
💾 Uploading 3239 structured chunks to the database...
✅ SUCCESS: The fully optimized RAG data is loaded into CloudNativePG!
